# Table A

Table A represents the cleaned, point-level time series of bin fill readings derived from an ultrasonic sensor dataset.

## Data Source
The dataset used in this project is the ultrasonic waste bin sensor dataset published on Zenodo:
https://zenodo.org/records/14988663

The dataset contains fill-level measurements captured by ultrasonic sensors installed in waste bins. Sensor timestamps are generated automatically, while collection timestamps are manually entered by service providers through the management system.

For this project, we use the corrected fill files provided by the dataset authors, which include preprocessing steps to address data quality issues present in the raw sensor readings.

## Table A Schema

Table A contains the following columns:

- ContainerID
- timestamp
- fill_percentage
- CIDX
- REC
- month_of_collection
- days_since_last_REC 
- is_weekend 

## Field Definitions
| Field | Description |
|------|-------------|
| ContainerID | Unique identifier for each waste bin.|
| timestamp | Date and time at which the fill-level reading was recorded by the sensor system. |
| fill_percentage | Percentage indicating how full the bin is at a given timestamp, based on the Mean fill-level value provided in the dataset.
| REC (Collection Reset Flag) | Indicates that a bin has just been emptied. A value of 1 marks the first fill reading immediately after a collection event; 0 otherwise. |
| CIDX (Cycle Index) | Identifies a single filling cycle between two collection events. All readings with the same CIDX belong to the same continuous filling period. |
| month_of_collection | Calendar month extracted from the timestamp, used to capture seasonal or monthly patterns in disposal behaviour. |
| days_since_last_REC | The number of days elapsed since the most recent collection event for the same bin. This value resets to zero after each collection and increases over time within a fill cycle. |
| is_weekend | Binary indicator showing whether the timestamp falls on a weekend (Saturday or Sunday), used to capture differences in disposal behaviour between weekdays and weekends. |


In [8]:
# Imports & Setup
from datasets import load_dataset
from huggingface_hub import list_repo_files
import pandas as pd
import os
import logging
from datasets.utils.logging import disable_progress_bar

# to suppress Hugging Face info messages due to missing yaml metadata
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
disable_progress_bar()

In [9]:
# Dataset Source
repo_id = "SA61team5/ultrasonic-waste-bin-sensor-raw"

all_files = list_repo_files(repo_id, repo_type="dataset")

# Select corrected fill csv files only
fill_files = [
    f for f in all_files 
    if "_fill_Corrected_with_metrics" in f and f.endswith(".csv")
]

# Limit to 30 bins for this prototype
fill_files = fill_files[:30]

## Per-bin Data Loading and Cleaning

For each selected bin file, we:
- Load the corrected fill data
- Extract the container identifier from the filename
- Standardise timestamps
- Rename and retain only relevant columns
- Calculate the number of days since the most recent collection event (days_since_last_REC)

In [13]:
all_fill_dfs = []

for file_path in fill_files:
    dataset = load_dataset(repo_id, data_files=file_path)
    df = pd.DataFrame(dataset["train"][:])

    # Extract container ID from filename
    container_id = os.path.basename(file_path).split("_")[1]
    df["ContainerID"] = container_id

    # Parse timestamp and derive month and is_weekend
    df["timestamp"] = pd.to_datetime(df["Date"])
    df["month_of_collection"] = df["timestamp"].dt.month
    df["is_weekend"] = df["timestamp"].dt.weekday >= 5
    df["is_weekend"] = df["is_weekend"].astype(int)

    df = df.rename(columns={"Mean": "fill_percentage"})
    df = df.dropna(subset=["fill_percentage"])

    df = df.dropna(subset=["Cidx"])

    df = df.drop(columns=["Max","Min","Fill","Date"])

    cols = ["ContainerID", "timestamp", "fill_percentage", "Cidx", "Rec", "month_of_collection", "is_weekend"]

    df = df[cols]

    all_fill_dfs.append(df)

final_fill_df = pd.concat(all_fill_dfs, ignore_index=True)

# Sort by bin ID and timestamp
final_fill_df = final_fill_df.sort_values(by=["ContainerID", "timestamp"]).reset_index(drop=True)

# Mark timestamps where a collection happened
final_fill_df["last_rec_timestamp"] = final_fill_df["timestamp"].where(
    final_fill_df["Rec"] == 1
)

# Forward-fill within each bin (record when the last collection happened in every row)
final_fill_df["last_rec_timestamp"] = (
    final_fill_df
    .groupby("ContainerID")["last_rec_timestamp"]
    .ffill()
)

# Compute days since last collection
final_fill_df["days_since_last_REC"] = (
    (final_fill_df["timestamp"] - final_fill_df["last_rec_timestamp"])
    .dt.total_seconds() / (60 * 60 * 24)
)

# Cleaning Data
final_fill_df["days_since_last_REC"] = (
    final_fill_df["days_since_last_REC"]
    .fillna(0)
    .clip(lower=0)
)

final_fill_df = final_fill_df.drop(columns=["last_rec_timestamp"])

In [14]:
print("Shape:", final_fill_df.shape)

from IPython.display import display

display(final_fill_df.head())
display(final_fill_df.tail())

Shape: (45835, 8)


,ContainerID,timestamp,fill_percentage,Cidx,Rec,month_of_collection,is_weekend,days_since_last_REC
0,1000,2021-01-16 11:30:00,52.5,0.0,1,1,1,0.000000
1,1000,2021-01-16 12:28:00,52.5,0.0,0,1,1,0.040278
2,1000,2021-01-16 13:28:00,52.5,0.0,0,1,1,0.081944
3,1000,2021-01-17 12:34:00,52.5,0.0,0,1,1,1.044444
4,1000,2021-01-18 12:41:00,52.5,0.0,0,1,0,2.049306


,ContainerID,timestamp,fill_percentage,Cidx,Rec,month_of_collection,is_weekend,days_since_last_REC
45830,10133,2023-08-24 12:24:00,75.0,29.0,0,8,0,6.945139
45831,10133,2023-08-24 14:18:00,81.0,29.0,0,8,0,7.024306
45832,10133,2023-08-24 15:15:00,100.0,29.0,0,8,0,7.063889
45833,10133,2023-08-25 13:57:00,3.0,30.0,1,8,0,0.000000
45834,10133,2023-08-28 14:13:00,77.0,30.0,0,8,0,3.011111


In [15]:
# save as csv file
final_fill_df.to_csv("../Cleaned-data/tableA.csv", index=False)